# Beach-volleyball exploratory analysis
Separate validation protocol; never opens test arrays. Source is team attack success rate, followed by frozen-latent blocks/digs readers. Provide the archived source/reuse prepared data. This is not the NBA/tennis optimizer protocol.

In [ ]:
from pathlib import Path
import json, subprocess, sys, shutil
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'code/train.py').is_file(), 'Open from repository root or notebooks/'
print('Repository:', ROOT)


In [ ]:
DATA=ROOT/'work/beach_prepared/data'
RUN=ROOT/'work/beach_run'
assert not RUN.exists(), 'Use a new folder or skip staging to resume'
for kind in ['source','reuse']:
    for name in ['fit.npz','val.npz','meta.json']:
        assert (DATA/kind/name).is_file(), str(DATA/kind/name)
shutil.copytree(ROOT/'extensions/beach_volleyball',RUN)
for kind in ['source','reuse']:
    (RUN/'data'/kind).mkdir(parents=True)
    for name in ['fit.npz','val.npz','meta.json']:
        shutil.copy2(DATA/kind/name,RUN/'data'/kind/name)

In [ ]:
for seed in [17,43,101]:
    subprocess.run([sys.executable,str(RUN/'run_job.py'),'finder','mse',str(seed)],cwd=RUN,check=True)
    for target in ['blocks_per_set','digs_per_set']:
        subprocess.run([sys.executable,str(RUN/'run_job.py'),'predictor','mse',str(seed),'--target',target],cwd=RUN,check=True)
subprocess.run([sys.executable,str(RUN/'baselines.py')],cwd=RUN,check=True)

In [ ]:
for p in sorted((RUN/'runs').glob('seed_*/mse/*/complete.json')):
    print(p.relative_to(RUN),json.loads(p.read_text()))
for p in sorted((RUN/'baselines').glob('*.json')):
    print(p.name,json.loads(p.read_text()))